# EDA — Walidacja danych źródłowych

Notebook sprawdza, czy pobrane pliki Criteo i Olist mają kolumny zgodne z założeniami staging modeli dbt.  
**Uruchom przed `make dbt-run`.** Nie musisz czytać całych 16M wierszy — bierzemy próbkę.

**Kolejność:**
1. Setup i ścieżki
2. Criteo — detekcja pliku, walidacja kolumn, statystyki
3. Olist — inwentaryzacja plików, kolumna `origin`, daty
4. Walidacja kluczy łączących (mql_id, seller_id)
5. Podsumowanie — gotowość do `dbt run`

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

DATA_DIR   = Path("../data")
CRITEO_DIR = DATA_DIR / "criteo"
OLIST_DIR  = DATA_DIR / "olist"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"DATA_DIR exists: {DATA_DIR.exists()}")
print(f"criteo/ contents: {list(CRITEO_DIR.iterdir()) if CRITEO_DIR.exists() else 'NOT FOUND'}")
print(f"olist/  contents: {[f.name for f in OLIST_DIR.iterdir()] if OLIST_DIR.exists() else 'NOT FOUND'}")

---
## 1. Criteo — detekcja pliku i próbka

In [ ]:
def detect_criteo_file() -> Path | None:
    for pattern in ["*.tsv.gz", "*.tsv", "*.csv.gz", "*.csv"]:
        matches = list(CRITEO_DIR.glob(pattern))
        if matches:
            return matches[0]
    return None

criteo_file = detect_criteo_file()
criteo_sample = None

if criteo_file is None:
    print("MISSING — brak pliku Criteo. Uruchom: make download")
else:
    is_tsv = "tsv" in criteo_file.name
    sep = "\t" if is_tsv else ","
    print(f"Found : {criteo_file.name}")
    print(f"Size  : {criteo_file.stat().st_size / 1_000_000:.1f} MB")
    print(f"Sep   : {'TAB' if is_tsv else 'COMMA'}")

    criteo_sample = pd.read_csv(
        criteo_file, sep=sep, nrows=100_000, low_memory=False
    )
    print(f"\nSample shape: {criteo_sample.shape}")
    print(f"\nActual columns ({len(criteo_sample.columns)}):")
    print(list(criteo_sample.columns))

### 1a. Walidacja mapowania kolumn vs. `stg_criteo__events.sql`

Kompletne mapowanie: wszystkie 22 kolumny pliku → aliasy w staging modelu.

> **Uwaga:** `timestamp` to względny czas (sekundy od początku datasetu), nie unix epoch.
> Criteo anonimizuje daty — `to_timestamp()` dałoby 1970-01-01. Używamy wartości surowej do sortowania.

In [ ]:
# Kompletne mapowanie: raw column -> alias w stg_criteo__events.sql
CRITEO_EXPECTED = {
    "uid":                   "user_id",
    "timestamp":             "click_timestamp_rel  (względny, nie unix epoch)",
    "campaign":              "campaign_id",
    "click":                 "is_click  (1=click, 0=impression)",
    "conversion":            "is_conversion_journey",
    "attribution":           "criteo_attribution  (Criteo własna atrybucja)",
    "conversion_id":         "conversion_id  (NULL gdy -1)",
    "conversion_timestamp":  "conversion_timestamp_rel  (NULL gdy -1)",
    "click_pos":             "click_position  (NULL gdy -1 = impression)",
    "click_nb":              "total_clicks_in_journey  (NULL gdy -1 = impression)",
    "cost":                  "click_cost",
    "cpo":                   "cost_per_order",
    "time_since_last_click": "seconds_since_last_click  (NULL gdy -1)",
    "cat1":                  "category_1",
    "cat2":                  "category_2",
    "cat3":                  "category_3",
    "cat4":                  "category_4",
    "cat5":                  "category_5",
    "cat6":                  "category_6",
    "cat7":                  "category_7",
    "cat8":                  "category_8",
    "cat9":                  "category_9",
}

if criteo_sample is not None:
    actual = set(criteo_sample.columns.str.lower())
    rows = []
    all_ok = True
    for raw_col, dbt_alias in CRITEO_EXPECTED.items():
        ok = raw_col in actual
        if not ok:
            all_ok = False
        rows.append({"raw_column": raw_col, "dbt_alias": dbt_alias, "status": "OK" if ok else "MISSING"})

    extra = sorted(actual - set(CRITEO_EXPECTED.keys()))

    result_df = pd.DataFrame(rows)
    display(result_df.style.apply(
        lambda s: ["background: #d4edda" if v == "OK" else "background: #f8d7da" for v in s],
        subset=["status"]
    ))

    if extra:
        print(f"\nNieznane kolumny w pliku (nie ma ich w staging): {extra}")

    print(f"\n{'OK — wszystkie kolumny pokryte w staging modelu.' if all_ok else 'UWAGA — staging model wymaga korekty!'}")
else:
    print("Brak danych Criteo — pomiń tę sekcję.")

### 1b. Statystyki kluczowych kolumn Criteo

> `click_nb = -1` dla 94.8% wierszy — to impresje, nie kliknięcia.  
> `fact_touchpoints` filtruje `WHERE is_click = 1`, więc tylko ~5% wierszy trafi do modeli atrybucji.

In [ ]:
if criteo_sample is not None:
    cols = criteo_sample.columns.str.lower()
    df = criteo_sample.copy()
    df.columns = cols

    print("=== Typy danych i null rate ===")
    info = pd.DataFrame({
        "dtype": df.dtypes,
        "null_count": df.isnull().sum(),
        "null_pct": (df.isnull().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    display(info)

In [ ]:
if criteo_sample is not None:
    df = criteo_sample.copy()
    df.columns = df.columns.str.lower()

    print("=== Click vs Impression split ===")
    if "click" in df.columns:
        vc = df["click"].value_counts()
        print(vc)
        print(f"Click rate: {vc.get(1, 0) / len(df) * 100:.2f}%")

    print("\n=== Conversion rate (z całego datasetu) ===")
    if "conversion" in df.columns:
        vc = df["conversion"].value_counts()
        print(vc)
        print(f"Conversion rate: {vc.get(1, 0) / len(df) * 100:.3f}%")

    print("\n=== Rozkład długości journey (click_nb) — tylko kliknięcia ===")
    if "click_nb" in df.columns:
        clicks_only = df[df["click"] == 1]["click_nb"]
        print(clicks_only.describe())
        print("\nTop 10 długości journey:")
        print(clicks_only.value_counts().head(10))

    print("\n=== Zakres timestamp (względny) ===")
    if "timestamp" in df.columns:
        ts = pd.to_numeric(df["timestamp"], errors="coerce")
        print(f"min: {ts.min()}  max: {ts.max()}  (zakres: {ts.max() - ts.min()} sekund = {(ts.max()-ts.min())/3600:.1f} godz)")
        print("UWAGA: to czas względny, nie unix epoch. Criteo anonimizuje daty.")

    print("\n=== Przykładowe wiersze (kliknięcia) ===")
    display(df[df["click"] == 1].head(3))

---
## 2. Olist — inwentaryzacja plików

In [ ]:
OLIST_FILES = {
    "olist_leads_qualified": "olist_marketing_qualified_leads_dataset.csv",
    "olist_leads_closed":    "olist_closed_deals_dataset.csv",
    "olist_orders":          "olist_orders_dataset.csv",
    "olist_order_items":     "olist_order_items_dataset.csv",
    "olist_order_payments":  "olist_order_payments_dataset.csv",
    "olist_customers":       "olist_customers_dataset.csv",
    "olist_sellers":         "olist_sellers_dataset.csv",
}

olist_dfs = {}
rows = []

for table, filename in OLIST_FILES.items():
    path = OLIST_DIR / filename
    if not path.exists():
        rows.append({"table": table, "file": filename, "status": "MISSING", "rows": None, "cols": None})
        continue
    df = pd.read_csv(path, low_memory=False)
    olist_dfs[table] = df
    rows.append({"table": table, "file": filename, "status": "OK", "rows": len(df), "cols": df.shape[1]})

inv = pd.DataFrame(rows)
display(inv.style.apply(
    lambda s: ["background: #d4edda" if v == "OK" else "background: #f8d7da" for v in s],
    subset=["status"]
))

### 2a. Kolumna `origin` — kanały pozyskania (kluczowe dla `dim_channel`)

In [ ]:
if "olist_leads_qualified" in olist_dfs:
    df = olist_dfs["olist_leads_qualified"]
    print(f"Kolumny: {list(df.columns)}")
    display(df.head(3))

    if "origin" in df.columns:
        origin_stats = (
            df["origin"]
            .str.lower().str.strip()
            .value_counts()
            .to_frame("count")
            .assign(pct=lambda x: (x["count"] / x["count"].sum() * 100).round(1))
        )
        print(f"\nKanały pozyskania (origin) — {df['origin'].nunique()} unikalnych:")
        display(origin_stats)
    else:
        print("UWAGA: brak kolumny 'origin'!")
else:
    print("Brak pliku olist_leads_qualified.")

### 2b. Olist orders — kolumny i zakres dat

In [ ]:
if "olist_orders" in olist_dfs:
    df = olist_dfs["olist_orders"]
    print(f"Kolumny: {list(df.columns)}")

    date_cols = [c for c in df.columns if "date" in c or "timestamp" in c]
    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors="coerce")
        print(f"  {col:45s}: {str(parsed.min())[:10]} → {str(parsed.max())[:10]}  (nulls: {parsed.isnull().sum()})")

    print(f"\norder_status distribution:")
    if "order_status" in df.columns:
        display(df["order_status"].value_counts())
else:
    print("Brak pliku olist_orders.")

### 2c. Walidacja kluczy łączących leads

> **Uwaga:** `seller_id` match w olist_sellers wynosi ~45% (380/842).  
> To znana cecha datasetu Olist — nie wszystkie closed leads mają odpowiadające rekordy sprzedawców.

In [ ]:
if "olist_leads_qualified" in olist_dfs and "olist_leads_closed" in olist_dfs:
    ql = olist_dfs["olist_leads_qualified"]
    cl = olist_dfs["olist_leads_closed"]

    print(f"leads_qualified: {len(ql):,} rows  |  kolumny: {list(ql.columns)}")
    print(f"leads_closed:    {len(cl):,} rows  |  kolumny: {list(cl.columns)}")

    if "mql_id" in ql.columns and "mql_id" in cl.columns:
        overlap = set(ql["mql_id"]) & set(cl["mql_id"])
        print(f"\nmql_id join: {len(overlap):,} wspólnych (conversion rate: {len(overlap)/len(ql)*100:.1f}%)")

    if "seller_id" in cl.columns:
        print(f"\nseller_id w leads_closed: {cl['seller_id'].nunique():,} unikalnych")
        if "olist_sellers" in olist_dfs:
            sellers = olist_dfs["olist_sellers"]
            if "seller_id" in sellers.columns:
                in_sellers = set(cl["seller_id"]) & set(sellers["seller_id"])
                pct = len(in_sellers) / cl["seller_id"].nunique() * 100
                print(f"seller_id match w olist_sellers: {len(in_sellers):,} / {cl['seller_id'].nunique():,} ({pct:.0f}%)")

    if "declared_monthly_revenue" in cl.columns:
        rev = pd.to_numeric(cl["declared_monthly_revenue"], errors="coerce")
        print(f"\ndeclared_monthly_revenue — min: {rev.min():.0f}  max: {rev.max():.0f}  nulls: {rev.isnull().sum()}")
else:
    print("Brak jednego z plików leads.")

### 2d. Walidacja kolumn payments i order_items

In [ ]:
for table in ["olist_order_payments", "olist_order_items"]:
    if table in olist_dfs:
        df = olist_dfs[table]
        print(f"=== {table} ===")
        print(f"Kolumny: {list(df.columns)}")
        display(df.head(3))
        print()

---
## 3. Podsumowanie — gotowość do `dbt run`

In [ ]:
checks = {}

# Criteo
checks["[Criteo] Plik znaleziony"] = criteo_file is not None
if criteo_sample is not None:
    actual = set(criteo_sample.columns.str.lower())
    required_criteo = set(CRITEO_EXPECTED.keys())
    missing = required_criteo - actual
    checks["[Criteo] Wszystkie kolumny obecne"] = len(missing) == 0
    if missing:
        print(f"  Brakuje: {missing}")

# Olist
required_tables = [
    "olist_leads_qualified",
    "olist_leads_closed",
    "olist_orders",
    "olist_order_items",
    "olist_order_payments",
]
for t in required_tables:
    checks[f"[Olist] {t}"] = t in olist_dfs

if "olist_leads_qualified" in olist_dfs:
    checks["[Olist] Kolumna 'origin' obecna"] = "origin" in olist_dfs["olist_leads_qualified"].columns

if "olist_leads_qualified" in olist_dfs and "olist_leads_closed" in olist_dfs:
    ql = olist_dfs["olist_leads_qualified"]
    cl = olist_dfs["olist_leads_closed"]
    checks["[Olist] mql_id join działa"] = "mql_id" in ql.columns and "mql_id" in cl.columns

print("=" * 65)
print("PODSUMOWANIE WALIDACJI")
print("=" * 65)
for check, ok in checks.items():
    icon = "OK " if ok else "FAIL"
    print(f"  [{icon}]  {check}")

all_pass = all(checks.values())
print("=" * 65)
if all_pass:
    print("Wszystko OK — mozesz uruchomic: make load && make dbt-run")
else:
    failed = [k for k, v in checks.items() if not v]
    print(f"Popraw {len(failed)} problem(y) przed uruchomieniem pipeline.")
    for f in failed:
        print(f"  - {f}")